# 多层感知机：整体流程介绍

这个小专题把原来的多层感知机案例拆成四个部分：

1. 整体流程介绍：先看网络结构、前向传播、反向传播和计算图之间的关系。
2. 数学推导：再把线性层、ReLU、Jacobian、梯度形状这些公式讲清楚。
3. $y=x^2$ 的计算过程：用一个样本手工算一遍前向、反向和参数更新。
4. $y=x^2$ 训练：最后用 NumPy 批量训练一个小网络拟合二次函数。

这里的目标不是追求复杂模型，而是把“神经网络节点图”和“矩阵公式”对齐起来。


## 案例目标

这个例子用一个很小的多层感知机拟合一维非线性函数：

$$
y = x^2
$$

网络结构是：

$$
1 \rightarrow 2 \rightarrow 1
$$

也就是 1 个输入、2 个隐藏层神经元、1 个输出。这个规模很小，适合手工推导前向传播和反向传播。

## 前向传播、反向传播与神经网络节点图

上面一行是神经网络的节点视角，下面一行是对应的向量化计算图。数学推导默认把单个样本写成列向量；中间的说明框给出两个视角的对应关系。

```mermaid
flowchart TB
    subgraph NN[神经网络节点视角]
        direction LR
        nx["输入层<br/>x"]
        nh1["隐藏节点 h1<br/>ReLU(w11*x + b11)"]
        nh2["隐藏节点 h2<br/>ReLU(w12*x + b12)"]
        no["输出层<br/>y_hat"]
        nx -- "w11" --> nh1
        nx -- "w12" --> nh2
        nh1 -- "w21" --> no
        nh2 -- "w22" --> no
    end

    map["映射关系<br/>输入层 x ↔ 列向量 x<br/>隐藏节点 h1 ↔ h[1]<br/>隐藏节点 h2 ↔ h[2]<br/>输出层 y_hat ↔ 列向量 y_hat"]

    subgraph CG[公式 / 计算图视角，列向量约定]
        direction LR
        x["输入 x"]
        z1["z1 = W1 x + b1<br/>W1=[[w11],[w12]], b1=[b11,b12]^T"]
        h["h = ReLU(z1)<br/>得到 [h1,h2]^T"]
        yhat["y_hat = W2 h + b2<br/>W2=[w21,w22], b2"]
        loss["L = 1/2 * (y_hat - y)^2<br/>真实值 y"]

        x --> z1 --> h --> yhat --> loss
        loss -. "dL/dy_hat = y_hat - y" .-> yhat
        yhat -. "dL/dh = W2.T * dL/dy_hat" .-> h
        h -. "dL/dz1 = dL/dh * ReLU'(z1)" .-> z1
        z1 -. "dL/dW1 = dL/dz1 * x.T, dL/db1 = dL/dz1" .-> x
        yhat -. "dL/dW2 = dL/dy_hat * h.T, dL/db2 = dL/dy_hat" .-> h
    end

    NN -.-> map
    map -.-> CG

    classDef data fill:#dff3ff,stroke:#277da1,color:#222;
    classDef op fill:#ecfdf3,stroke:#2b9348,color:#222;
    classDef lossStyle fill:#ffe3e3,stroke:#c92a2a,color:#222;
    classDef neuron fill:#f3e8ff,stroke:#7b2cbf,color:#222;
    classDef mapStyle fill:#f8f9fa,stroke:#adb5bd,color:#222;
    class x,nx data;
    class z1,h,yhat op;
    class loss lossStyle;
    class nh1,nh2,no neuron;
    class map mapStyle;
```
